In [1]:
# Cell 1: Environment check & file listing
import os, sys

# Show what numpy/torch versions Kaggle provides out of the box
import numpy as np
import torch
print(f'numpy: {np.__version__}')
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import pandas as pd
print(f'pandas: {pd.__version__}')

# List input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/dhriti1004/video-ds/query.jpeg
/kaggle/input/datasets/dhriti1004/video-ds/vid1.mp4
/kaggle/input/datasets/dhriti1004/video-dataset/query2.jpeg
/kaggle/input/datasets/dhriti1004/video-dataset/vid_data/vid_data/vid.mp4.mp4


In [ ]:
import json
import os

# --- WEB APP CONFIG INJECTION ---
# This safely reads the user's dropdown choice from the dataset.
config_path = '/kaggle/input/forensic-input/config.json'
MODEL_VERSION = 'Baseline (V1)'

if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        data = json.load(f)
        MODEL_VERSION = data.get('model_version', 'Baseline (V1)')

print(f"Executing pipeline with version: {MODEL_VERSION}")
# ---------------------------------


In [2]:
# Cell 2: Install only packages NOT pre-installed on Kaggle
import os, sys

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    # DO NOT touch numpy - Kaggle's numpy 2.0.2 is required by pandas/opencv/etc.
    # Install genuinely missing packages with --no-deps for facenet to avoid numpy conflict
    print('Installing missing packages...')
    r1 = os.system(f'{sys.executable} -m pip install -q --no-deps facenet-pytorch')
    r2 = os.system(f'{sys.executable} -m pip install -q ultralytics faiss-cpu lapx reportlab')
    print(f'Install done (r1={r1}, r2={r2})')

    # PIL._util patch for torchvision compatibility
    try:
        import pathlib, PIL._util as _u
        if not hasattr(_u, 'is_directory'):
            _u.is_directory = lambda f: isinstance(f, (str, pathlib.Path)) and os.path.isdir(f)
        if not hasattr(_u, 'is_path'):
            _u.is_path = lambda f: isinstance(f, (str, pathlib.Path))
        print('PIL patch OK')
    except Exception as e:
        print(f'PIL patch skipped: {e}')
else:
    print('Local run - skipping setup')


In [3]:
import os, cv2, json, math, warnings
print('cv2 OK')
import numpy as np
print(f'numpy {np.__version__} OK')
import pandas as pd
print('pandas OK')
import torch
print(f'torch {torch.__version__} OK')
import torchvision.transforms as T
import torchvision.models as models
print('torchvision OK')
from facenet_pytorch import MTCNN, InceptionResnetV1
print('facenet_pytorch OK')
from ultralytics import YOLO
print('ultralytics OK')
import faiss
print('faiss OK')
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
print('All imports OK')

# directory structure
BASE_DIR = "/kaggle/working/forensic_pipeline"
DIRS = {
    "meta": os.path.join(BASE_DIR, "metadata"),
    "track": os.path.join(BASE_DIR, "tracking"),
    "extract": os.path.join(BASE_DIR, "extraction"),
    "cluster": os.path.join(BASE_DIR, "clustering"),
    "faiss": os.path.join(BASE_DIR, "faiss_indices"),
    "output": os.path.join(BASE_DIR, "final_output")
}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

#find video
def get_video_path(base="/kaggle/input"):
    vids = []
    for r, _, f in os.walk(base):
        for file in f:
            if file.lower().endswith((".mp4", ".avi", ".mkv")):
                vids.append((os.path.getsize(os.path.join(r, file)), os.path.join(r, file)))
    if not vids: raise FileNotFoundError("No video found in /kaggle/input")
    return sorted(vids, reverse=True)[0][1]

VIDEO_PATH = get_video_path()

# Auto-detect GPU
# Detect GPU compatibility: torch 2.10 requires sm_70+
# Kaggle may assign P100 (sm_60) which is INCOMPATIBLE with torch 2.10 CUDA ops
if torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability()
    if _cap[0] >= 7:  # sm_70+ supported
        DEVICE = torch.device('cuda')
    else:
        DEVICE = torch.device('cpu')
        print(f'GPU sm_{_cap[0]}{_cap[1]} not supported by torch {torch.__version__} (needs sm_70+), using CPU')
else:
    DEVICE = torch.device('cpu')

if DEVICE.type == 'cuda':
    print(f"GPU active: {torch.cuda.get_device_name(0)}")
print(f"Setup Complete. Using Video: {VIDEO_PATH} on {DEVICE}")


In [4]:
#metadata extraction
meta_path = os.path.join(DIRS["meta"], "frame_meta.csv")
if os.path.exists(meta_path):
    print("Loading checkpoint for Temporal & Quality Metadata...")
    frame_meta = pd.read_csv(meta_path)
    print(f"Loaded {len(frame_meta)} frames from checkpoint.")
else:
    print("Extracting Temporal & Quality Metadata...")
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

    rows = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        #forensic time
        pos_msec = cap.get(cv2.CAP_PROP_POS_MSEC)
        time_sec = (pos_msec / 1000.0) if pos_msec > 0 else (frame_idx / fps)
        
        #blur score
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        
        rows.append({"frame_index": frame_idx, "time_sec": time_sec, "laplacian": lap_var})
        frame_idx += 1

    cap.release()

    frame_meta = pd.DataFrame(rows)
    frame_meta.to_csv(meta_path, index=False)
    print(f"Processed {len(frame_meta)} frames.")

In [5]:
#yolo
track_path = os.path.join(DIRS["track"], "tracked_persons.csv")
if os.path.exists(track_path):
    print("Loading checkpoint for YOLO Tracking...")
    track_df = pd.read_csv(track_path)
    print(f"Loaded {len(track_df)} tracked bounding boxes from checkpoint.")
else:
    print("Running Detection & Tracking...")
    model = YOLO("yolov8n.pt")
    cap = cv2.VideoCapture(VIDEO_PATH)

    lap_map = dict(zip(frame_meta["frame_index"], frame_meta["laplacian"]))
    track_rows = []
    frame_idx = 0

    #read video
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        results = model.track(frame, classes=[0], conf=0.15, iou=0.4, persist=True, tracker="bytetrack.yaml", verbose=False, device=DEVICE.type)

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy()
            confs = results[0].boxes.conf.cpu().numpy()
            
            for box, tid, conf in zip(boxes, ids, confs):
                track_rows.append({
                    "frame_index": frame_idx,
                    "track_id": int(tid),
                    "x1": int(box[0]), "y1": int(box[1]), "x2": int(box[2]), "y2": int(box[3]),
                    "yolo_conf": float(conf),
                    "laplacian": lap_map.get(frame_idx, 0.0)
                })
                
        frame_idx += 1

    cap.release()
    track_df = pd.DataFrame(track_rows)
    track_df.to_csv(track_path, index=False)
    print(f"Logged {len(track_df)} tracked bounding boxes.")

In [6]:
#keyframe extraction
extract_meta_path = os.path.join(DIRS["extract"], "extraction_meta.csv")
if os.path.exists(extract_meta_path):
    print("Loading checkpoint for Keyframe Extraction...")
    body_vecs = np.load(os.path.join(DIRS["extract"], "body_vecs.npy"))
    effnet_vecs = np.load(os.path.join(DIRS["extract"], "effnet_vecs.npy"))
    face_vecs = np.load(os.path.join(DIRS["extract"], "face_vecs.npy"))
    extracted_meta = pd.read_csv(extract_meta_path).to_dict('records')
    print(f"Loaded {len(body_vecs)} keyframe sets from checkpoint.")
else:
    print("Initializing Extraction Models...")
    mtcnn = MTCNN(keep_all=False, device=DEVICE)
    face_model = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)

    resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    body_model = torch.nn.Sequential(*list(resnet.children())[:-1]).eval().to(DEVICE)

    effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    effnet_model = torch.nn.Sequential(*list(effnet.children())[:-1]).eval().to(DEVICE)

    to_tensor = T.Compose([T.ToPILImage(), T.Resize((256, 128)), T.ToTensor(), 
                           T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    # pre-calculate
    print("Calculating Keyframe Triggers...")
    frames_to_read = defaultdict(list)
    OPTIMAL_LAP = 300.0

    for tid, group in track_df.groupby("track_id"):
        last_center, last_frame = None, None
        for _, r in group.sort_values("frame_index").iterrows():
            frm, lap = int(r["frame_index"]), r["laplacian"]
            if lap < 100.0: continue 
            
            x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]
            cx, cy = (x1+x2)/2, (y1+y2)/2
            
            time_trig = (last_frame is None) or (frm - last_frame >= 30)
            disp_trig = False
            if last_center:
                diag = math.hypot(x2-x1, y2-y1) + 1e-6
                disp_trig = (math.hypot(cx-last_center[0], cy-last_center[1]) / diag) >= 0.10
                
            if time_trig or disp_trig:
                frames_to_read[frm].append({"tid": tid, "bbox": (x1,y1,x2,y2), "y_conf": r["yolo_conf"], "lap": lap})
                last_center, last_frame = (cx, cy), frm

    #sequential processing
    print(f"Extracting features from {len(frames_to_read)} unique frames...")
    cap = cv2.VideoCapture(VIDEO_PATH)
    curr_frame = 0

    extracted_meta = []
    face_vecs, body_vecs, effnet_vecs = [], [], []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        if curr_frame in frames_to_read:
            for tgt in frames_to_read[curr_frame]:
                x1, y1, x2, y2 = map(int, tgt["bbox"])
                y_conf, lap, tid = tgt["y_conf"], tgt["lap"], tgt["tid"]
                
                h, w = frame.shape[:2]
                body_crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
                face_crop = frame[max(0, y1):min(h, y1+int(0.5*(y2-y1))), max(0, x1):min(w, x2)]
                
                if body_crop.size == 0: continue
                    
                # body & EffNet
                with torch.no_grad():
                    t = to_tensor(body_crop).unsqueeze(0).to(DEVICE)
                    b_vec = body_model(t).view(-1).cpu().numpy()
                    e_vec = effnet_model(t).view(-1).cpu().numpy()
                b_vec = b_vec / (np.linalg.norm(b_vec) + 1e-8)
                e_vec = e_vec / (np.linalg.norm(e_vec) + 1e-8)
                
                #face
                f_vec, f_conf = None, 0.0
                if face_crop.shape[0] > 40:
                    try:
                        rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
                        f_tens, p = mtcnn(rgb, return_prob=True)
                        if f_tens is not None:
                            with torch.no_grad():
                                f_vec = face_model(f_tens.unsqueeze(0).to(DEVICE)).view(-1).cpu().numpy()
                                f_vec = f_vec / (np.linalg.norm(f_vec) + 1e-8)
                            f_conf = float(p)
                    except: pass
                    
                # dynamic confidence scoring
                sharp_ratio = min(1.0, lap / OPTIMAL_LAP)
                b_conf = float(y_conf * sharp_ratio)
                e_conf = float(y_conf * min(1.0, (lap + 50) / OPTIMAL_LAP)) # EffNet is robust to slight blur
                
                idx = len(body_vecs)
                body_vecs.append(b_vec)
                effnet_vecs.append(e_vec)
                face_vecs.append(f_vec if f_vec is not None else np.zeros(512, dtype=np.float32))
                
                extracted_meta.append({
                    "emb_idx": idx, "track_id": tid, "frame": curr_frame,
                    "has_face": (f_vec is not None),
                    "face_conf": f_conf, "body_conf": b_conf, "effnet_conf": e_conf,
                    "laplacian": lap, "yolo_conf": y_conf
                })
                
        curr_frame += 1
    cap.release()

    # Save
    if len(body_vecs) > 0:
        np.save(os.path.join(DIRS["extract"], "body_vecs.npy"), np.stack(body_vecs))
        np.save(os.path.join(DIRS["extract"], "effnet_vecs.npy"), np.stack(effnet_vecs))
        np.save(os.path.join(DIRS["extract"], "face_vecs.npy"), np.stack(face_vecs))
    else:
        np.save(os.path.join(DIRS["extract"], "body_vecs.npy"), np.empty((0, 2048), dtype=np.float32))
        np.save(os.path.join(DIRS["extract"], "effnet_vecs.npy"), np.empty((0, 1280), dtype=np.float32))
        np.save(os.path.join(DIRS["extract"], "face_vecs.npy"), np.empty((0, 512), dtype=np.float32))
        
    pd.DataFrame(extracted_meta).to_csv(extract_meta_path, index=False)
    print(f"Extracted {len(body_vecs)} keyframe sets.")

In [7]:
#clustering
proto_meta_path = os.path.join(DIRS["cluster"], "proto_meta.csv")
if os.path.exists(proto_meta_path):
    print("Loading checkpoint for Identity Prototypes...")
    proto_meta = pd.read_csv(proto_meta_path).to_dict('records')
    p_bodies = np.load(os.path.join(DIRS["cluster"], "proto_body.npy"))
    p_effnets = np.load(os.path.join(DIRS["cluster"], "proto_effnet.npy"))
    p_faces = np.load(os.path.join(DIRS["cluster"], "proto_face.npy"))
    print(f"Loaded {len(p_bodies)} unique person prototypes from checkpoint.")
else:
    print("Building Identity Prototypes...")
    ext_meta = pd.read_csv(os.path.join(DIRS["extract"], "extraction_meta.csv"))
    b_vecs = np.load(os.path.join(DIRS["extract"], "body_vecs.npy"))
    e_vecs = np.load(os.path.join(DIRS["extract"], "effnet_vecs.npy"))
    f_vecs = np.load(os.path.join(DIRS["extract"], "face_vecs.npy"))

    proto_meta, p_bodies, p_effnets, p_faces = [], [], [], []

    if len(ext_meta) > 0:
        for tid, group in ext_meta.groupby("track_id"):
            indices = group["emb_idx"].values
            
            # Body & EffNet Protos (Mean of all valid keyframes)
            b_proto = b_vecs[indices].mean(axis=0)
            e_proto = e_vecs[indices].mean(axis=0)
            p_bodies.append(b_proto / (np.linalg.norm(b_proto) + 1e-8))
            p_effnets.append(e_proto / (np.linalg.norm(e_proto) + 1e-8))
            
            face_idx = group[group["has_face"] == True]["emb_idx"].values
            if len(face_idx) > 0:
                f_proto = f_vecs[face_idx].mean(axis=0)
                p_faces.append(f_proto / (np.linalg.norm(f_proto) + 1e-8))
                has_f = True
            else:
                p_faces.append(np.zeros(512, dtype=np.float32))
                has_f = False
                
            best_idx = group.loc[group["laplacian"].idxmax()]
            proto_meta.append({
                "person_id": int(tid), 
                "proto_idx": len(p_bodies) - 1,
                "rep_frame": int(best_idx["frame"]),
                "face_conf": float(group["face_conf"].max()), 
                "body_conf": float(group["body_conf"].mean()),
                "effnet_conf": float(group["effnet_conf"].mean()),
                "has_face": has_f
            })

    #save
    if len(p_bodies) > 0:
        np.save(os.path.join(DIRS["cluster"], "proto_body.npy"), np.stack(p_bodies))
        np.save(os.path.join(DIRS["cluster"], "proto_effnet.npy"), np.stack(p_effnets))
        np.save(os.path.join(DIRS["cluster"], "proto_face.npy"), np.stack(p_faces))
    else:
        np.save(os.path.join(DIRS["cluster"], "proto_body.npy"), np.empty((0, 2048), dtype=np.float32))
        np.save(os.path.join(DIRS["cluster"], "proto_effnet.npy"), np.empty((0, 1280), dtype=np.float32))
        np.save(os.path.join(DIRS["cluster"], "proto_face.npy"), np.empty((0, 512), dtype=np.float32))
        
    pd.DataFrame(proto_meta).to_csv(proto_meta_path, index=False)
    print(f"Found {len(p_bodies)} unique person prototypes.")

In [8]:
#faiss indices
body_idx_path = os.path.join(DIRS["faiss"], "body.index")
effnet_idx_path = os.path.join(DIRS["faiss"], "effnet.index")
face_idx_path = os.path.join(DIRS["faiss"], "face.index")

if os.path.exists(body_idx_path) and os.path.exists(effnet_idx_path) and os.path.exists(face_idx_path):
    print("Loading checkpoint for FAISS Indices...")
    faiss_body = faiss.read_index(body_idx_path)
    faiss_effnet = faiss.read_index(effnet_idx_path)
    faiss_face = faiss.read_index(face_idx_path)
    print("Loaded 3 vector databases from checkpoint.")
else:
    print("Constructing FAISS Indices...")
    p_bodies = np.load(os.path.join(DIRS["cluster"], "proto_body.npy")).astype('float32')
    p_effnets = np.load(os.path.join(DIRS["cluster"], "proto_effnet.npy")).astype('float32')
    p_faces = np.load(os.path.join(DIRS["cluster"], "proto_face.npy")).astype('float32')

    # CPU-only FAISS (GPU FAISS skipped due to P100 sm_60 incompatibility)
    faiss_body = faiss.IndexFlatIP(p_bodies.shape[1])
    faiss_effnet = faiss.IndexFlatIP(p_effnets.shape[1])
    faiss_face = faiss.IndexFlatIP(p_faces.shape[1])

    if len(p_bodies) > 0:
        faiss_body.add(p_bodies)
        faiss_effnet.add(p_effnets)
        faiss_face.add(p_faces)

    faiss.write_index(faiss_body, body_idx_path)
    faiss.write_index(faiss_effnet, effnet_idx_path)
    faiss.write_index(faiss_face, face_idx_path)

    print("3 vector databases built.")

In [9]:
import os
import cv2
import faiss
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#retrieve and plot
def query_forensic_database(query_img_path, top_k=5):
    print(f"\nSearching database for {query_img_path} ---")
    img = cv2.imread(query_img_path)
    if img is None:
        print(f"Could not read query image at {query_img_path}")
        return None
   
    global mtcnn, face_model, body_model, effnet_model, to_tensor
    
    # Initialize models if they don't exist globally (e.g. if checkpoint was loaded)
    if 'mtcnn' not in globals() or mtcnn is None:
        from facenet_pytorch import MTCNN, InceptionResnetV1
        import torchvision.transforms as T
        import torchvision.models as models
        
        print("Initializing Retrieval Models for Query Processing...")
        mtcnn = MTCNN(keep_all=False, device=DEVICE)
        face_model = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
        
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        body_model = torch.nn.Sequential(*list(resnet.children())[:-1]).eval().to(DEVICE)
        
        effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        effnet_model = torch.nn.Sequential(*list(effnet.children())[:-1]).eval().to(DEVICE)
        
        to_tensor = T.Compose([T.ToPILImage(), T.Resize((256, 128)), T.ToTensor(), 
                               T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    q_b_vec, q_e_vec, q_f_vec = None, None, None
    q_f_conf = 0.0
    
    with torch.no_grad():
        t = to_tensor(img).unsqueeze(0).to(DEVICE)
        q_b_vec = body_model(t).view(1, -1).cpu().numpy().astype('float32')
        q_e_vec = effnet_model(t).view(1, -1).cpu().numpy().astype('float32')
        q_b_vec, q_e_vec = q_b_vec/np.linalg.norm(q_b_vec), q_e_vec/np.linalg.norm(q_e_vec)
        
    try:
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        f_tens, prob = mtcnn(rgb, return_prob=True)
        if f_tens is not None:
            with torch.no_grad():
                q_f_vec = face_model(f_tens.unsqueeze(0).to(DEVICE)).view(1, -1).cpu().numpy().astype('float32')
                q_f_vec = q_f_vec/np.linalg.norm(q_f_vec)
            q_f_conf = float(prob)
    except: pass

    # Search all three indices for candidate matches
    idx_b = faiss.read_index(os.path.join(DIRS["faiss"], "body.index"))
    idx_e = faiss.read_index(os.path.join(DIRS["faiss"], "effnet.index"))
    idx_f = faiss.read_index(os.path.join(DIRS["faiss"], "face.index"))
    
    # 1. Search Body Index
    dist_b, ind_b = idx_b.search(q_b_vec, top_k) 
    candidate_ids = list(ind_b[0])
    
    # 2. Search General Embedding Index (EffNet)
    dist_e, ind_e = idx_e.search(q_e_vec, top_k)
    for e_id in ind_e[0]:
        if e_id not in candidate_ids and e_id >= 0:
            candidate_ids.append(e_id)
            
    # 3. Search Face Index
    if q_f_vec is not None:
        dist_f, ind_f = idx_f.search(q_f_vec, top_k)
        for f_id in ind_f[0]:
            if f_id not in candidate_ids and f_id >= 0:
                candidate_ids.append(f_id)
    
    # rerank
    p_meta = pd.read_csv(os.path.join(DIRS["cluster"], "proto_meta.csv"))
    results = []
    
    for c_id in candidate_ids:
        if c_id < 0: continue
        c_row = p_meta[p_meta["proto_idx"] == c_id].iloc[0]
        
        # Prioritize face match heavily if both query and candidate have faces
        if q_f_vec is not None and c_row["has_face"]:
            w_f = 0.70
            w_b = 0.15
            w_e = 0.15
        else:
            w_f = 0.0
            w_b = 0.50
            w_e = 0.50
            
        q_f_safe = q_f_vec if q_f_vec is not None else np.zeros((1, 512), dtype='float32')
        s_b = idx_b.reconstruct(int(c_id)).reshape(1, -1).dot(q_b_vec.T)[0][0]
        s_e = idx_e.reconstruct(int(c_id)).reshape(1, -1).dot(q_e_vec.T)[0][0]
        s_f = idx_f.reconstruct(int(c_id)).reshape(1, -1).dot(q_f_safe.T)[0][0] if c_row["has_face"] else 0.0
        
        final_score = (w_f * s_f) + (w_b * s_b) + (w_e * s_e)
        
        results.append({
            "person_id": int(c_row["person_id"]),
            "rep_frame": int(c_row["rep_frame"]),
            "final_score": final_score,
            "weights": (w_f, w_b, w_e)
        })
        
    results = sorted(results, key=lambda x: x["final_score"], reverse=True)
    best = results[0]
    
    print(f"Top match: Person {best['person_id']} (Score: {best['final_score']:.4f})")
    
    track_df = pd.read_csv(os.path.join(DIRS["track"], "tracked_persons.csv"))
    bbox_row = track_df[(track_df["track_id"] == best['person_id']) & \
                        (track_df["frame_index"] == best['rep_frame'])]
    
    if len(bbox_row) > 0:
        bbox = bbox_row.iloc[0]
        x1, y1, x2, y2 = int(bbox['x1']), int(bbox['y1']), int(bbox['x2']), int(bbox['y2'])
        
        v_cap = cv2.VideoCapture(VIDEO_PATH)
        v_cap.set(cv2.CAP_PROP_POS_FRAMES, best['rep_frame'])
        ret, frame = v_cap.read()
        v_cap.release()
        
        if ret:
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 4)
            cv2.putText(frame, f"Match: {best['final_score']:.2f}", (x1, max(0, y1-10)), \
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 3)
            
            query_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            match_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            axes[0].imshow(query_rgb)
            axes[0].set_title("Query Image", fontsize=14, fontweight='bold')
            axes[0].axis("off")
            axes[1].imshow(match_rgb)
            axes[1].set_title(f"Target(Person ID: {best['person_id']})", fontsize=14, fontweight='bold')
            axes[1].axis("off")
            plt.tight_layout()
            plt.show()
    
    return best

def generate_evidence(best_match_dict):
    frame_idx = int(best_match_dict["rep_frame"])
    pid = int(best_match_dict["person_id"])
    
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    track_df = pd.read_csv(os.path.join(DIRS["track"], "tracked_persons.csv"))
    person_tracks = track_df[track_df["track_id"] == pid]
    
    start_f = max(0, int(frame_idx - (fps * 3.0)))
    end_f = int(frame_idx + (fps * 6.0))
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
    ret, sample = cap.read()
    if not ret: return
        
    out_path = os.path.join(DIRS["output"], f"evidence_failover_p{pid}.mp4")
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (sample.shape[1], sample.shape[0]))
    
    print(f"\nWriting clip (Frames {start_f} to {end_f})...")
    
    last_box = None
    template_img = None
    
    for f in range(start_f, end_f + 1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, f)
        ret, frame = cap.read()
        if not ret: continue
        
        b_row = person_tracks[person_tracks["frame_index"] == f]
        
        if len(b_row) > 0:
            bx = b_row.iloc[0]
            x1, y1, x2, y2 = int(bx['x1']), int(bx['y1']), int(bx['x2']), int(bx['y2'])
            last_box = (x1, y1, x2, y2)
            
            template_img = frame[max(0,y1):min(frame.shape[0],y2), max(0,x1):min(frame.shape[1],x2)].copy()
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(frame, f"Target: ID {pid}", (x1, max(0, y1-10)), \
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
        else:
            if template_img is not None and template_img.size > 0 and last_box is not None:
                x1, y1, x2, y2 = last_box
                pad = 50                 
                sx1 = max(0, x1 - pad)
                sy1 = max(0, y1 - pad)
                sx2 = min(frame.shape[1], x2 + pad)
                sy2 = min(frame.shape[0], y2 + pad)
                search_region = frame[sy1:sy2, sx1:sx2]
                
                if search_region.shape[0] >= template_img.shape[0] and search_region.shape[1] >= template_img.shape[1]:
                    res = cv2.matchTemplate(search_region, template_img, cv2.TM_CCOEFF_NORMED)
                    _, max_val, _, max_loc = cv2.minMaxLoc(res)
                    
                    if max_val > 0.35: 
                        nx1 = sx1 + max_loc[0]
                        ny1 = sy1 + max_loc[1]
                        nx2 = nx1 + template_img.shape[1]
                        ny2 = ny1 + template_img.shape[0]
                        
                        cv2.rectangle(frame, (nx1, ny1), (nx2, ny2), (0, 165, 255), 3) 
                        cv2.putText(frame, "Target", (nx1, max(0, ny1-10)), \
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
                        
                        last_box = (nx1, ny1, nx2, ny2)
        
        time_sec = f / fps
        cv2.putText(frame, f"Time: {time_sec:.2f}s | Golden Frame: {frame_idx}", (10, 30), \
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        out.write(frame)
            
    out.release()
    cap.release()
    print(f"Clip Generated at: {out_path}")

# dynamically find query image from /kaggle/input if it exists
def get_query_image_path(base="/kaggle/input"):
    for r, _, f in os.walk(base):
        for file in f:
            if file.lower().endswith((".jpeg", ".jpg", ".png")):
                return os.path.join(r, file)
    return "/kaggle/input/datasets/dhriti1004/video-dataset/query2.jpeg" # Fallback

QUERY_IMAGE = get_query_image_path()

best_match_result = query_forensic_database(QUERY_IMAGE)

if best_match_result is not None:
    generate_evidence(best_match_result)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

# --- OPTIMIZATION, SYNOPSIS, AND METRICS MODULE ---\n
def get_optimized_intervals(track_df, best_person_id, min_score_threshold=0.3):
    person_df = track_df[track_df['track_id'] == best_person_id].copy()
    if person_df.empty: return []
    person_df = person_df.sort_values('frame_index').reset_index(drop=True)
    
    max_lap = track_df['laplacian'].max() if track_df['laplacian'].max() > 0 else 1.0
    person_df['saliency'] = (person_df['yolo_conf'] * 0.6) + ((person_df['laplacian'] / max_lap) * 0.4)
    person_df['smoothed'] = person_df['saliency'].rolling(window=15, min_periods=1, center=True).mean()
    
    saliency_array = person_df['smoothed'].fillna(0).values
    peaks, _ = find_peaks(saliency_array, height=min_score_threshold, distance=30)
    
    intervals = []
    for p in peaks:
        start_idx, end_idx = p, p
        while start_idx > 0 and saliency_array[start_idx] > (min_score_threshold * 0.5): start_idx -= 1
        while end_idx < len(saliency_array) - 1 and saliency_array[end_idx] > (min_score_threshold * 0.5): end_idx += 1
        intervals.append((int(person_df.loc[start_idx, 'frame_index']), int(person_df.loc[end_idx, 'frame_index'])))
        
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]] if intervals else []
    for current in intervals[1:]:
        last = merged[-1]
        if current[0] <= last[1] + 15: merged[-1] = (last[0], max(last[1], current[1]))
        else: merged.append(current)
    return merged

def create_final_synopsis(optimized_intervals, video_path, target_id, output_dir, track_df):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out_path = os.path.join(output_dir, \"final_synopsis.mp4\")
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*\"mp4v\"), fps, (w, h))
    
    # Filter track data for target person once to speed up lookups
    person_tracks = track_df[track_df[\"track_id\"] == target_id]
    
    for start_f, end_f in sorted(optimized_intervals):
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
        for f in range(start_f, end_f + 1):
            ret, frame = cap.read()
            if not ret: break
            
            # Find and draw bounding box for this frame
            b_row = person_tracks[person_tracks[\"frame_index\"] == f]
            if len(b_row) > 0:
                bx = b_row.iloc[0]
                x1, y1, x2, y2 = int(bx['x1']), int(bx['y1']), int(bx['x2']), int(bx['y2'])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                cv2.putText(frame, f\"Target: ID {target_id}\", (x1, max(0, y1-10)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            cv2.putText(frame, f\"ID: {target_id} | Time: {f/fps:.2f}s\", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            out.write(frame)
    out.release(); cap.release()
    return out_path

def calculate_evaluation_metrics(orig_path, syn_path, gt_events, intervals):
    cap_o = cv2.VideoCapture(orig_path); cap_s = cv2.VideoCapture(syn_path)
    fps = cap_o.get(cv2.CAP_PROP_FPS)
    cr = cap_o.get(cv2.CAP_PROP_FRAME_COUNT) / cap_s.get(cv2.CAP_PROP_FRAME_COUNT)
    cap_o.release(); cap_s.release()
    captured_seconds = [(s/fps, e/fps) for s, e in intervals]
    recall = sum(1 for gt in gt_events if any(c_s <= gt[1] and c_e >= gt[0] for c_s, c_e in captured_seconds)) / len(gt_events)
    return {\"Compression Ratio\": round(cr, 2), \"Event Recall (%)\": round(recall * 100, 2)}

# --- EXECUTION ---
best_match = best_match_result
track_df = pd.read_csv(os.path.join(DIRS[\"track\"], \"tracked_persons.csv\"))
intervals = get_optimized_intervals(track_df, best_match['person_id'])
if len(intervals) == 0:
    print('No highly salient intervals found for synopsis. Skipping synopsis generation.')
    syn_path = 'N/A'\n    metrics = {'Compression Ratio': 0.0, 'Event Recall (%)': 0.0}
else:
    syn_path = create_final_synopsis(intervals, VIDEO_PATH, best_match['person_id'], DIRS[\"output\"], track_df)
    metrics = calculate_evaluation_metrics(VIDEO_PATH, syn_path, [(10.0, 15.0), (30.0, 40.0)], intervals)

# Save job metadata for audio reconstruction on local machine
import json
top_matches_serializable = []
for res in results[:10]:
    top_matches_serializable.append({
        \"person_id\": int(res[\"person_id\"]),
        \"rep_frame\": int(res[\"rep_frame\"]),
        \"final_score\": float(res[\"final_score\"]),
        \"weights\": [float(w) for w in res[\"weights\"]]
    })

job_meta = {
    \"target_id\": int(best_match['person_id']),
    \"rep_frame\": int(best_match['rep_frame']),
    \"fps\": float(cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS)),
    \"intervals\": [[int(start), int(end)] for start, end in intervals] if intervals else [],
    \"top_matches\": top_matches_serializable
}
with open(os.path.join(DIRS[\"output\"], \"job_meta.json\"), \"w\") as f:
    json.dump(job_meta, f)

print(f\"\\n--- REPORT ---\\n{metrics}\\nSynopsis saved: {syn_path}\")

In [ ]:
import shutil
import os

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    print("Zipping outputs for easy download...")
    # Zip the forensic_pipeline directory
    shutil.make_archive("/kaggle/working/forensic_pipeline_results", 'zip', "/kaggle/working/forensic_pipeline")
    print("All checkpoints and outputs saved to /kaggle/working/forensic_pipeline_results.zip")
    print("You can download this zip file from the Data/Output section of your Kaggle notebook.")
else:
    print("Running locally. Outputs are already saved in the local directory.")